In [83]:
import httpx
import os
import sys
import json
import pandas as pd

sys.path.append(os.path.abspath('./src'))
from db_functions import DotaDB
opendota_url = 'https://api.opendota.com/api'
db = DotaDB()

In [ ]:
result = httpx.get(f'{opendota_url}/live').json()
pro_matches = []
for r in result:
    if r['league_id'] != 0:
        pro_matches.append(r)
pd.DataFrame(pro_matches).iloc[:, :15]

,activate_time,deactivate_time,server_steam_id,lobby_id,league_id,lobby_type,game_time,delay,spectators,game_mode,average_mmr,match_id,series_id,team_name_radiant,team_name_dire
0,1774853855,0,90283662945047562,29773684374794677,18867,1,1446,120,1,2,0,8750122430,1080822,Nethercore,Teiko
1,1774856366,1774859770,90283667901656075,29773684396987958,18867,1,2732,120,1,2,0,8750150836,1080822,Nethercore,Teiko
2,1774858358,1774861044,90283661343804435,29773684403631623,19255,1,1783,900,12,2,0,8750161286,1080836,_PowerRangers,VP.Prodigy
3,1774859737,1774863705,90283666646696979,29773684416754826,19520,1,3083,900,19,2,0,8750183428,1080838,NGNB,GreenTeaWatch
4,1774860558,0,90283667755473943,29773684430915901,18867,1,-49,120,1,2,0,8750203717,1080845,Teiko,Shinigami Gaming
5,1774861662,1774863783,90283661106707479,29773684447463007,18867,1,1938,120,2,1,0,8750218681,1080848,Teiko,Shinigami Gaming
6,1774864246,0,90283668738875421,29773684463252089,19520,1,1278,900,12,2,0,8750240751,1080838,NGNB,GreenTeaWatch
7,1774863949,0,90283669218320403,29773684470466339,18867,1,592,120,1,2,0,8750249409,1080848,Shinigami Gaming,Teiko


In [ ]:
with httpx.Client(headers=db.stratz_headers) as client:
    query = '''
    query($id: Long!) {
        match(id: $id) {
            id 
            startDateTime
        }
    }
    '''
    result = db.query_stratz(client, query, variables={'id': 8757271594})

In [ ]:
result

{'data': {'match': {'id': 8757271594, 'startDateTime': 1775305743}}}

In [85]:
with httpx.Client() as client:
    result = db.query_opendota(client, endpoint=f'matches/{8760126026}')

In [86]:
result

{'version': 22,
 'match_id': 8760126026,
 'teamfights': [{'start': 318,
   'end': 360,
   'last_death': 345,
   'deaths': 3,
   'players': [{'deaths_pos': {},
     'ability_uses': {'weaver_shukuchi': 3, 'weaver_the_swarm': 1},
     'ability_targets': {},
     'item_uses': {'madstone_bundle': 1, 'faerie_fire': 1, 'magic_wand': 1},
     'killed': {},
     'deaths': 0,
     'buybacks': 0,
     'damage': 454,
     'healing': 421,
     'gold_delta': 224,
     'xp_delta': 492,
     'xp_start': 1309,
     'xp_end': 1801},
    {'deaths_pos': {'126': {'128': 1}},
     'ability_uses': {'huskar_inner_fire': 1},
     'ability_targets': {},
     'item_uses': {'tpscroll': 1, 'magic_stick': 1, 'faerie_fire': 1},
     'killed': {'npc_dota_hero_death_prophet': 1},
     'deaths': 1,
     'buybacks': 0,
     'damage': 1177,
     'healing': 115,
     'gold_delta': 394,
     'xp_delta': 126,
     'xp_start': 2243,
     'xp_end': 2369},
    {'deaths_pos': {},
     'ability_uses': {'centaur_double_edge': 1, 

In [7]:
query = 'SELECT * FROM hero_details'
heroes = db.query_select_to_df(query, table='hero_details')

In [100]:
match_wards

[{'match_id': 8760094725,
  'hero_id': 23,
  'time': 391,
  'type': 1,
  'positionX': 129.9,
  'positionY': 124.2},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': -58,
  'type': 0,
  'positionX': 126,
  'positionY': 155.4},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 445,
  'type': 0,
  'positionX': 192.3,
  'positionY': 85.3},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 750,
  'type': 0,
  'positionX': 155,
  'positionY': 73.7},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 1223,
  'type': 0,
  'positionX': 136.1,
  'positionY': 155.7},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 1247,
  'type': 0,
  'positionX': 164,
  'positionY': 133.8},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 1341,
  'type': 0,
  'positionX': 142.1,
  'positionY': 169.9},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 35,
  'type': 1,
  'positionX': 95.5,
  'positionY': 164.1},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 445,
  'type': 1,
  'posit

In [ ]:
match_death_events = []
mid = result['match_id']
for p in result['players']:
    hero_id = p['hero_id']
    for kill in p['kills_log']:
        match_death_events.append(
            {
                'match_id': mid,
                'hero_id': int(heroes[heroes['name'] == kill['key']].get('id').iloc[0]),
                'time': kill['time'],
                'attacker': hero_id
            }
        )

In [99]:
pd.DataFrame(match_details, index=[0])

,id,tournamentId,tournamentRound,leagueId,radiantTeamId,direTeamId,seriesId,clusterId,didRadiantWin,startDateTime,...,towerStatusRadiant,towerStatusDire,barracksStatusRadiant,barracksStatusDire,rank,actualRank,averageRank,averageImp,radiant_score,dire_score
0,8760126026,None,None,18866,8680909,9872558,1083445,272,True,1775466314,...,1846,0,63,0,None,None,None,None,34,8


In [88]:
match_details = {
    'id': result['match_id'],
    'tournamentId': result.get('tournament_id'),
    'tournamentRound': result.get('tournament_round'),
    'leagueId': result['leagueid'],
    'radiantTeamId': result['radiant_team_id'], 
    'direTeamId': result['dire_team_id'],
    'seriesId': result['series_id'],
    'clusterId': result['cluster'],
    'didRadiantWin': result['radiant_win'],
    'startDateTime': result['start_time'],
    'endDateTime': result['start_time'] + result['duration'],
    'durationSeconds': result['duration'],
    'firstBloodTime': result['first_blood_time'],
    'towerStatusRadiant': result['tower_status_radiant'],
    'towerStatusDire': result['tower_status_dire'],
    'barracksStatusRadiant': result['barracks_status_radiant'],
    'barracksStatusDire': result['barracks_status_dire'],
    'rank': result.get('rank_tier'),
    'actualRank': result.get('rank_tier_actual'),
    'averageRank': result.get('average_rank'),
    'averageImp': result.get('average_imp'),
    'radiant_score': result['radiant_score'],
    'dire_score': result['dire_score']
}

In [ ]:
match_pick_bans = []
mid = result['match_id']
for mpb in result['picks_bans']:
    match_pick_bans.append(
        {
            'match_id': mid,
            'isPick': mpb['is_pick'],
            'heroId': mpb['hero_id'],
            'order': mpb['order'],
            'isRadiant': mpb['team'] == 0
        }
    )

In [ ]:
match_players = []
mid = result['match_id']
for p in result['players']:
    is_radiant = p['team_number'] == 0
    if (is_radiant and p['team_number'] == 0) or (not is_radiant and p['team_number'] == 1):
        is_victory = True
    else:
        is_victory = False           
    match_players.append(
        {
            'match_id': mid,
            'heroId': p['hero_id'],
            'isRadiant': is_radiant,
            'isVictory': is_victory,
            'variant': p['hero_variant'],
            'networth': p['net_worth'],
            'goldPerMinute': p['gold_per_min'],
            'goldSpent': p['gold_spent'],
            'towerDamage': p['tower_damage'],
            'heroDamage': p['hero_damage'],
            'steamAccountId': p['account_id'],
            'partyId': p['party_id'],
            'name': p['name'],
            'kills': p['kills'],
            'deaths': p['deaths'],
            'assists': p['assists']
        }
    )

In [56]:
query = 'SELECT * FROM item_details_opendota'
items = db.query_select_to_df(query, table='item_details_opendota')

In [ ]:
match_purchases = []
mid = result['match_id']
for p in result['players']:
    hero_id = p['hero_id']
    for pur in p['purchase_log']:
        match_purchases.append(
            {
                'match_id': mid,
                'hero_id': hero_id,
                'time': pur['time'],
                'itemId': items[items['shortName'] == pur['key']].get('id').iloc[0]
            }
        )

In [ ]:
rune_map = {
    "0": "DOUBLE_DAMAGE",
    "1": "HASTE",
    "2": "ILLUSION",
    "3": "INVISIBILITY",
    "4": "REGEN",
    "5": "BOUNTY",
    "6": "ARCANE",
    "7": "WATER",
    "8": "WISDOM",
    "9": "SHIELD"
}
match_runes = []
mid = result['match_id']
for p in result['players']:
    for rune in p['runes_log']:
        match_runes.append(
            {
                'match_id': mid,
                'hero_id': p['hero_id'],
                'time': rune['time'],
                'rune': rune_map[rune['key']]
            }
        )

In [66]:
query = 'SELECT * FROM npcs'
npcs = db.query_select_to_df(query, table='npcs')

In [71]:
obj

{'time': 674,
 'type': 'building_kill',
 'unit': 'npc_dota_creep_badguys_melee',
 'key': 'npc_dota_goodguys_tower1_bot'}

In [87]:
match_tower_deaths = []
mid = result['match_id']
for obj in result['objectives']:
    if obj['type'] == 'building_kill':
        try:
            attacker = heroes[heroes['name'] == obj['unit']].get('id').iloc[0]
        except:
            attacker = 'non-hero'
        match_tower_deaths.append(
            {
                'match_id': mid,
                'time': obj['time'],
                'npcId': npcs[npcs['name'] == obj['key']].get('id').iloc[0],
                'isRadiant': 'goodguys' in obj['key'],
                'attacker': attacker
            }
        )

In [ ]:
match_wards = []
for p in result['players']:
    for ward in p['obs_log']:
        match_wards.append(
            {
                'match_id': mid,
                'hero_id': p['hero_id'],
                'time': ward['time'],
                'type': 0,
                'positionX': ward['x'],
                'positionY': ward['y']
            }
        )
    for ward in p['sen_log']:
        match_wards.append(
            {
                'match_id': mid,
                'hero_id': p['hero_id'],
                'time': ward['time'],
                'type': 1,
                'positionX': ward['x'],
                'positionY': ward['y']
            }
        )

In [77]:
match_wards

[{'match_id': 8760094725,
  'hero_id': 23,
  'time': 391,
  'type': 1,
  'positionX': 129.9,
  'positionY': 124.2},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': -58,
  'type': 0,
  'positionX': 126,
  'positionY': 155.4},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 445,
  'type': 0,
  'positionX': 192.3,
  'positionY': 85.3},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 750,
  'type': 0,
  'positionX': 155,
  'positionY': 73.7},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 1223,
  'type': 0,
  'positionX': 136.1,
  'positionY': 155.7},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 1247,
  'type': 0,
  'positionX': 164,
  'positionY': 133.8},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 1341,
  'type': 0,
  'positionX': 142.1,
  'positionY': 169.9},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 35,
  'type': 1,
  'positionX': 95.5,
  'positionY': 164.1},
 {'match_id': 8760094725,
  'hero_id': 88,
  'time': 445,
  'type': 1,
  'posit

In [ ]:
result

{'version': 22,
 'match_id': 8760094725,
 'teamfights': [{'start': -15,
   'end': 20,
   'last_death': 5,
   'deaths': 3,
   'players': [{'deaths_pos': {},
     'ability_uses': {'dawnbreaker_converge': 1,
      'dawnbreaker_celestial_hammer': 1},
     'ability_targets': {},
     'item_uses': {'tpscroll': 1},
     'killed': {},
     'deaths': 0,
     'buybacks': 0,
     'damage': 279,
     'healing': 0,
     'gold_delta': 110,
     'xp_delta': 25},
    {'deaths_pos': {'138': {'100': 1}},
     'ability_uses': {},
     'ability_targets': {},
     'item_uses': {'tpscroll': 1,
      'magic_stick': 1,
      'quelling_blade': 2,
      'faerie_fire': 1},
     'killed': {},
     'deaths': 1,
     'buybacks': 0,
     'damage': 363,
     'healing': 194,
     'gold_delta': 95,
     'xp_delta': 25},
    {'deaths_pos': {},
     'ability_uses': {},
     'ability_targets': {},
     'item_uses': {},
     'killed': {},
     'deaths': 0,
     'buybacks': 0,
     'damage': 0,
     'healing': 0,
     'gold

In [15]:
with httpx.Client() as client:
    schema = db.query_opendota(client, endpoint='schema')